In [1]:
import os
import json

import numpy as np
import pandas as pd

In [ ]:
runs = [
    os.path.join('..', 'experiments', str(x)) for x in range(1585, 1617)
]

assert len(runs) == 32, len(runs)

In [3]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [4]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))

    assert (info['do_masking'] and info['add_bc']) or (not info['add_bc'])
    
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # print(log['training_time'].cumsum().max())
    
    if os.path.exists(os.path.join(run, 'init_time.txt')):
        init_time = float(open(os.path.join(run, 'init_time.txt')).read())
    else:
        init_time = 0.
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] <= min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item() + init_time
    
    # epoch where we first reach 5% relative error
    # first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    # info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item() + init_time
    
    df.loc[len(df)] = info

In [5]:
df['min_rel_l2_error'] = df['min_rel_l2_error'] * 100

df = df.round(3)

for col in df.columns:
    df[col] = df[col].astype(str)

In [6]:
results = df[[
    'soln_name',
    'n_train',
    # 'time_to_5p_rel_l2_error',
    'min_rel_l2_error',
    'time_to_min_rel_l2_error',
]].rename(columns={
    # 'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'n_train': '# points'
})

results

,Solution,# points,Min. RL2E. (%),Time to min. RL2E. (s)
0,Plane waves,100,2.643,520.401
1,Plane waves,200,1.006,61.375
2,Plane waves,500,1.067,19.208
3,Plane waves,1000,1.007,39.591
4,Plane waves,2000,1.079,34.989
5,Plane waves,5000,1.041,33.182
6,Plane waves,10000,1.061,40.436
7,Plane waves,12000,1.28,37.692
8,Radial waves,100,7.371,137.951
9,Radial waves,200,2.929,144.574


In [7]:
print(results.to_latex())

\begin{tabular}{lllll}
\toprule
 & Solution & # points & Min. RL2E. (%) & Time to min. RL2E. (s) \\
\midrule
0 & Plane waves & 100 & 2.643 & 520.401 \\
1 & Plane waves & 200 & 1.006 & 61.375 \\
2 & Plane waves & 500 & 1.067 & 19.208 \\
3 & Plane waves & 1000 & 1.007 & 39.591 \\
4 & Plane waves & 2000 & 1.079 & 34.989 \\
5 & Plane waves & 5000 & 1.041 & 33.182 \\
6 & Plane waves & 10000 & 1.061 & 40.436 \\
7 & Plane waves & 12000 & 1.28 & 37.692 \\
8 & Radial waves & 100 & 7.371 & 137.951 \\
9 & Radial waves & 200 & 2.929 & 144.574 \\
10 & Radial waves & 500 & 1.012 & 164.654 \\
11 & Radial waves & 1000 & 1.003 & 105.763 \\
12 & Radial waves & 2000 & 1.026 & 86.571 \\
13 & Radial waves & 5000 & 1.002 & 94.494 \\
14 & Radial waves & 10000 & 1.047 & 97.262 \\
15 & Radial waves & 12000 & 1.023 & 109.639 \\
16 & Hopf fibration & 100 & 1.035 & 33.196 \\
17 & Hopf fibration & 200 & 1.103 & 14.155 \\
18 & Hopf fibration & 500 & 1.027 & 11.496 \\
19 & Hopf fibration & 1000 & 1.079 & 12.512 \\
2